# Does this even make sense?

In [1]:
########## INIT ####################################################################################
import pickle, os, traceback, json
from collections import deque
from copy import deepcopy
from pprint import pprint
from typing import Deque

import numpy as np
import matplotlib.pyplot as plt

from magpie_control.ur5 import _CAMERA_XFORM

from aspire.env_config import env_var
from aspire.symbols import GraspObj, ObjPose, euclidean_distance_between_symbols, extract_pose_as_homog
from aspire.BlocksTask import set_blocks_env

from TaskPlanner import set_experiment_env
from draw_jupyter import set_render_env, render_memory_list, render_state_and_plan_step
from utils import deep_copy_memory_list


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
[Open3D INFO] Resetting default logger to print to terminal.


## Setup Iteration

In [2]:
######### CONSTANTS ###############################################################################

_DATA_DRIVE = "STARGAZER/DATA_TANK"

_PLOT_DIR   = "/media/james/FILEPILE/EROM/data/plots/"
_GC_CYCLE   = False 
_F_EXTRACT  = f"{_PLOT_DIR}outData.pkl"
_T_EXTRACT  = f"{_PLOT_DIR}outText.json"

_MIN_STATE_SIZE_BYTES = 500.0

_TITLE_FONT_SIZE = 13
_TIGHT_MARGIN    =  0.05

_DEFAULT_DIV = 100 #80 #100 #200

tests = [
    "KC-KP",
    "SC-KP",
    "KC-SP",
    "SC-SP",
]

longTestNames = [
    "Known Class & Known Pose", 
    "Sensed Class & Known Pose", 
    "Known Class & Sensed Pose", 
    "Sensed Class & Sensed Pose", 
]

datasets = [
    [ f"/media/james/{_DATA_DRIVE}/2025-08B_{test}" for test in tests ],
    [ f"/media/james/{_DATA_DRIVE}/RWB_2025-09_{test}" for test in tests ],
]

dataLabels = ["RGB", "RBW",]
datNamLong = {
    "RGB": "Red-Green-Blue", 
    "RBW": "Red-Black-White",
}

blcNam = {
    "RGB": ['redBlock','grnBlock','bluBlock',], 
    "RBW": ['redBlock','blkBlock','whtBlock',],
}

eBlcNam = {
    "RGB": ['redBlock', 'grnBlock', 'bluBlock', env_var("_NULL_NAME"),], 
    "RBW": ['redBlock', 'blkBlock', 'whtBlock', env_var("_NULL_NAME"),],
}

plotExt = ".pdf"

_MISC_DIR = "/media/james/STARGAZER/DATA_TANK/misc_data/" 
_SIM_INFO_PATH = f"{_MISC_DIR}SimInfo.pkl" 

## Iterate Episoses

In [3]:
from EROM.utils import print_header
from EROM.Reader import EROM_Reader

In [4]:
totRes = dict() # Input Data 
totPrb = dict() # Output Metrics


### For every block set ###
for iii, paths in enumerate( datasets ):

    setNam = dataLabels[iii]
    suffix = "_" + setNam
    skip   = False

    totRes[ setNam ] = dict()

    ### For every scenario ###
    for ii, test in enumerate( tests ):

        if ii < 1:
            continue
        
        totRes[ setNam ][ test ] = deque()

        print_header( f"TEST, {setNam}: {test}", preWidth = 10, totWidth = 100, capitalize = True )

        ##### Init ####################################################
        path     = paths[ii]
        longTNam = longTestNames[ii]

        testRecord = [os.path.join( path, item ) for item in sorted( os.listdir( path ) ) if ((".pkl" in f"{item}".lower()) and ("_OCV-State" not in f"{item}") and ("thin" not in f"{item}".lower()))]
        trueRecord = [os.path.join( path, item ) for item in sorted( os.listdir( path ) ) if ((".pkl" in f"{item}".lower()) and ("_OCV-State" in f"{item}")     and ("thin" not in f"{item}".lower()))]

        print( f"{len(testRecord)} test records" )
        print( f"{len(trueRecord)} true records" )

        ### For every episode ###
        for _i_, episodePath in enumerate( testRecord ):
            # print( f"\n{episodePath}, {int(os.path.getsize(episodePath)/1e6)}MB" )
            try:
                reader = EROM_Reader( episodePath, suppressLoad = True )
            except RuntimeError:
                print( f"\nSKIPPED: {episodePath}\n" )
                continue

            assocStates, assocSteps = reader.get_states_and_steps()

            for i in range( len( assocStates ) ):
                fState_i = assocStates[i]
                fStep_i  = assocSteps[i]

                with open( fState_i, 'rb' ) as f:
                    state_i = pickle.load(f)

                """
                "labels" : list(), #- List of objects in this scene
                "image"  : dict(), #- Lookup of color images used
                "depth"  : dict(), #- Lookup of depth images used
                "clouds" : deque(), # Collection of clouds obtained from the masked images
        
                "objects": deque(), # Collection of readings obtained from the masked images
                
                "sensed" : list(), # Collection of symbols obtained from the robot
                "symbols": dict(), #- Lookup of objects obtained from the readings
                """
        
                with open( fStep_i, 'rb' ) as f:
                    step_i = pickle.load(f)
                
                """ [ ..., ['msg', 't', 'data'], ... ] """ 

                # render_memory_list( syms = list( state_i["symbols"].values() ) )
                render_state_and_plan_step( 
                    syms    = list( state_i["symbols"].values() ), 
                    planStr = reader.step_plan_from_thin_step( step_i )
                )
                
                print( reader.planning_result_from_thin_step( step_i, prntPlan = True ) )
                
                
            print( '#'*75 )
            # break # Per episode

        break

    break
            






########## TEST, RGB: SC-KP #####################################################################
21 test records
269 true records


Renderer(camera=PerspectiveCamera(aspect=1.618, far=10.0, fov=70.0, position=(-0.5680000000000001, -0.61874999…


['Move Free from [[-1 0 0 -0.58835]\n'
 ' [0 1 0 -0.12629]\n'
 ' [0 0 -1 0.55303]\n'
 ' [0 0 0 1]] --to-> [[-1 0 0 -0.384]\n'
 ' [0 1 0 -0.0715]\n'
 ' [0 0 -1 0.25]\n'
 ' [0 0 0 1]]',
 'Pick grnBlock at [[1 0 0 -0.38]\n'
 ' [0 1 0 -0.077]\n'
 ' [0 0 1 0.02]\n'
 ' [0 0 0 1]] from table',
 'Move Holding grnBlock --to-> <ObjPose 0, Vec: [-0.2 -0.3 0.02] >',
 'Place grnBlock at [[1 0 0 -0.2]\n'
 ' [0 1 0 -0.3]\n'
 ' [0 0 1 0.02]\n'
 ' [0 0 0 1]] onto table']

True


Renderer(camera=PerspectiveCamera(aspect=1.618, far=10.0, fov=70.0, position=(-0.5680000000000001, -0.61874999…


['Move Free from [[-1 0 0 -0.57703]\n'
 ' [0 1 0 0.10145]\n'
 ' [0 0 -1 0.53161]\n'
 ' [0 0 0 1]] --to-> [[-1 0 0 -0.384]\n'
 ' [0 1 0 0.0335]\n'
 ' [0 0 -1 0.25]\n'
 ' [0 0 0 1]]',
 'Pick redBlock at [[1 0 0 -0.38]\n'
 ' [0 1 0 0.028]\n'
 ' [0 0 1 0.02]\n'
 ' [0 0 0 1]] from table',
 'Move Holding redBlock --to-> <ObjPose 123, Vec: [-0.2 -0.3 0.06] >',
 'Stack redBlock at [[1 0 0 -0.2]\n'
 ' [0 1 0 -0.3]\n'
 ' [0 0 1 0.06]\n'
 ' [0 0 0 1]] onto grnBlock']

True


...

False


Renderer(camera=PerspectiveCamera(aspect=1.618, far=10.0, fov=70.0, position=(-0.5680000000000001, -0.61874999…


['Move Free from [[-1 0 0 -0.20978]\n'
 ' [0 1 0 -0.243]\n'
 ' [0 0 -1 0.69824]\n'
 ' [0 0 0 1]] --to-> [[-1 0 0 -0.38427]\n'
 ' [0 1 0 -0.17176]\n'
 ' [0 0 -1 0.2592]\n'
 ' [0 0 0 1]]',
 'Pick bluBlock at [[0.05203 -0.94594 -0.32016 -0.38027]\n'
 ' [-0.98708 -5.5455e-05 -0.16025 -0.17726]\n'
 ' [0.15157 0.32436 -0.93371 0.029197]\n'
 ' [0 0 0 1]] from table',
 'Move Holding bluBlock --to-> <ObjPose 239, Vec: [-0.20163 -0.3074 0.11201] >',
 'Stack bluBlock at [[0.05203 -0.94594 -0.32016 -0.20163]\n'
 ' [-0.98708 -5.5455e-05 -0.16025 -0.3074]\n'
 ' [0.15157 0.32436 -0.93371 0.11201]\n'
 ' [0 0 0 1]] onto redBlock']

True


Renderer(camera=PerspectiveCamera(aspect=1.618, far=10.0, fov=70.0, position=(-0.5680000000000001, -0.61874999…

True
###########################################################################
